# Project 4 — Schema-Guided Receipt Extraction

This starter uses CORD v2 OCR to identify four receipt-summary fields: `subtotal`, `service_charge`, `tax`, and `total_amount`. Every other available OCR line is labeled `OTHER`. The goal is a small, understandable baseline that leaves value normalization, layout features, richer schemas, and human feedback workflows for your investigation.

### Problem Formulation

The goal of this project is to **extract structured information from receipts**, such as the subtotal, service charge, tax, and total amount.

A complete system would need to identify the relevant information in the receipt, extract the values, and populate a structured form.

In this initial baseline, we simplify this problem to **OCR-line classification**. This doesn't need to be the actual problem formulation. You should be able to improve it based on your research.

CORD v2 already provides the text extracted from receipt images using OCR. We treat each OCR line as an independent classification example:

- **Input:** The text of one OCR line (e.g., `TOTAL 45,500`).
- **Output:** One of five labels: `subtotal`, `service_charge`, `tax`, `total_amount`, or `OTHER`.

Our classifier uses only the OCR text, without considering the line's position, surrounding text, or the original image.

**This is only one component of a complete information-extraction system.** Classifying a line as `total_amount` does not necessarily mean we have correctly extracted and normalized the corresponding value.

#### Evaluating the Baseline

The baseline evaluates predictions **at the OCR-line level**, using standard classification metrics such as accuracy and F1-score.

However, the actual application operates at the **receipt level**: we ultimately want to extract the correct values for each receipt. Consider this when reflecting on the tasks below.


## 1. Your Tasks

The provided notebook establishes a simple baseline for detecting potentially toxic comments. Your task is to **evaluate this baseline, identify its limitations, and investigate possible improvements**.

### 1.1 Explore the Dataset

Get familiar with the data and the classification task.

- Examine the dataset characteristics, class distribution, and representative examples.
- Consider how the target labels are defined and whether the data is suitable for the intended application.
- Identify potential issues related to data quality, coverage, or representativeness.

### 1.2 Evaluate the Baseline

Determine how well the model performs and which evaluation criteria are appropriate.

- Select and justify metrics relevant to the content moderation task.
- Evaluate overall and class-wise performance.
- Consider the consequences of different types of errors, particularly false positives and false negatives.

### 1.3 Analyze Errors and Diagnose Limitations

Investigate where and why the classifier makes mistakes.

- Examine misclassified examples and identify recurring patterns.
- Formulate possible explanations for the observed errors.
- Distinguish observations supported by the data from hypotheses that require further investigation.

### 1.4 Investigate Possible Improvements

Based on your observations and relevant literature, identify promising directions for improving the baseline.

- Research alternative approaches, representations, or model configurations.
- Explain which limitations these approaches might address.
- Where feasible, implement and evaluate an improvement against the original baseline using a consistent evaluation setup.

### 1.5 Document Your Findings

Summarize your experiments and the reasoning behind your decisions.

- What did you investigate, and why?
- What evidence supports your observations and conclusions?
- Which improvements appear promising, and what remains unresolved?

**Important:** Your decisions should be grounded in **empirical evidence or relevant research**, rather than assumptions. When proposing changes, explain the motivation and evaluate their effects whenever possible.

Use the validation set for model development and comparisons. Keep the test set reserved for final evaluation.



## 2. Starter Code

### 2.1 Load and Inspect the Data

Before running the notebook, download and prepare the CORD v2 dataset using the following commands.

**1. Download the dataset:**

```bash
python scripts/download_dataset.py "naver-clova-ix/cord-v2" "data/cord_v2"
```

**2. Prepare a reproducible sample:**

```bash
python project_baselines/project_04/scripts/prepare_cord.py \
  --source-mode local \
  --train-size 100 \
  --validation-size 50 \
  --test-size 50 \
  --seed 42
```

For this baseline, we focus on identifying **receipt summary fields**, rather than individual purchased items.

Each OCR line is assigned one of five labels:

- `subtotal`
- `service_charge`
- `tax`
- `total_amount`
- `OTHER` — Any OCR line that does not correspond to one of the four target fields.

The prepared dataset will be used to train and evaluate our OCR-line classifier.

In [ ]:
from pathlib import Path
import os
import pandas as pd

# 1. Locate the project directory
# The notebook can be run from the repository root or the project folder.
project_dir = Path("project_baselines/project_04")
if not project_dir.exists():
    project_dir = Path(".")

# Locate the prepared dataset (optionally using a custom data directory)
data_dir = Path(os.getenv(
    "CORD_PREPARED_DIR",
    project_dir / "data" / "prepared" / "cord_v2"
))

# 2. Load the prepared OCR data
# Each row represents one OCR line from a receipt.
# 'text' is the model input; 'label' is the target field.
lines = pd.read_csv(data_dir / "ocr_lines.csv")

# Replace missing OCR text with empty strings
lines["text"] = lines["text"].fillna("")

# 3. Inspect the dataset
display(lines.sample(8, random_state=42))

# Examine the distribution of labels across partitions
display(pd.crosstab(lines["label"], lines["split"]))

In [ ]:
pd.crosstab(lines["label"], lines["split"], margins=True)

### 2.2 Inspect one receipt

The images and richer annotations remain available separately. They are useful for inspection, but they are not model features in this baseline.

In [ ]:
import json
from IPython.display import display
from PIL import Image

# Load receipt metadata, including the paths to the original images
with (data_dir / "documents.jsonl").open(encoding="utf-8") as f:
    documents = {doc["document_id"]: doc for doc in map(json.loads, f)}

# Select one receipt from the training set
example_id = lines.loc[lines["split"] == "train", "document_id"].iloc[0]
example = documents[example_id]

# Display the original receipt image
display(Image.open(data_dir / example["image"]))

# Display the corresponding OCR lines and their reference labels
display(lines[lines["document_id"] == example_id][["text", "label"]])


### 2.3 TF-IDF + logistic regression baseline

The official document partitions are preserved: all lines from a receipt stay in the same split. We train on `train`, evaluate on `validation`, and do not use `test`. Only OCR text is passed to the model.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

train = lines[lines["split"] == "train"].copy()
validation = lines[lines["split"] == "validation"].copy()

baseline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
    ("classifier", LogisticRegression(
        class_weight="balanced", max_iter=1000, random_state=42
    )),
])
baseline.fit(train["text"], train["label"])
validation_predictions = baseline.predict(validation["text"] )

### 2.4 Initial evaluation

Accuracy is influenced by the large `OTHER` class, so also inspect macro-F1 and the per-class report.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, f1_score

print(f"Accuracy: {accuracy_score(validation['label'], validation_predictions):.3f}")
print(f"Macro-F1: {f1_score(validation['label'], validation_predictions, average='macro'):.3f}")
print(classification_report(validation["label"], validation_predictions, zero_division=0))

## 2.5 A minimal extraction illustration

For one receipt, the code below shows the OCR line with the highest model score for each target field. This is only a candidate selector: finding a line is not the same as extracting and normalizing the correct numeric value, and an absent field should not necessarily receive a value.

In [ ]:
target_fields = ["subtotal", "service_charge", "tax", "total_amount"]
receipt_id = validation["document_id"].iloc[0]
receipt_lines = validation[validation["document_id"] == receipt_id].copy()
probabilities = pd.DataFrame(
    baseline.predict_proba(receipt_lines["text"]),
    columns=baseline.named_steps["classifier"].classes_,
    index=receipt_lines.index,
)

candidates = []
for field in target_fields:
    row_index = probabilities[field].idxmax()
    candidates.append({
        "field": field,
        "candidate_text": receipt_lines.loc[row_index, "text"],
        "score": probabilities.loc[row_index, field],
    })

pd.DataFrame(candidates)

## 3. Your experiments